# Echelon Chess Engine - Colab Training (C++ Optimized)

This notebook trains the Echelon chess engine using high-performance C++ backend for AlphaZero-style self-play.

**Before running:**
1. Go to Runtime > Change runtime type > GPU (T4)
2. Run all cells in order

In [ ]:
# Clone repository
!git lfs install
!git clone https://github.com/falloficarus22/echelon.git
%cd echelon
!git lfs pull

In [ ]:
!ls -lh checkpoints/*.pt

In [ ]:
# Install dependencies
!pip install pybind11 -q

In [ ]:
# Remove any pre-compiled .so files (they won't work on Colab)
!rm -f cpp/*.so *.so
print("Cleaned up pre-compiled binaries")

In [ ]:
# Compile high-performance C++ backend
import sys
import sysconfig

# Get the correct extension suffix
ext_suffix = sysconfig.get_config_var('EXT_SUFFIX')
print(f"Python version: {sys.version}")
print(f"Extension suffix: {ext_suffix}")
print(f"GLIBC version: ", end="")
!ldd --version | head -n1
print("\nCompiling C++ backend...")

# Compile
!cd cpp && g++ -O3 -Wall -shared -std=c++17 -fPIC \
    $(python3 -m pybind11 --includes) \
    attacks.cpp magic.cpp board.cpp mcts.cpp echelon_cpp_wrapper.cpp \
    -o echelon_cpp{ext_suffix}

# Verify the file was created
!ls -lh cpp/echelon_cpp*
print("\n✓ C++ backend compiled successfully!")

In [ ]:
# Verify the module loads correctly
import sys
sys.path.insert(0, './cpp')

try:
    import echelon_cpp
    print(f"Module location: {echelon_cpp.__file__}")
    echelon_cpp.init()
    print("✓ C++ module loaded successfully!")
    print("✓ Attack tables initialized!")
except Exception as e:
    print(f"✗ Error loading module: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Training

**Performance on T4 GPU:**
- Self-play: ~2-10 seconds per game (vs 30-60s in pure Python)
- Training: Limited by neural network, not chess engine
- Total speedup: ~60x faster than pure Python

The script will run 10 iterations with 5 games each.

In [ ]:
# Start training using C++ backend
!python train_cpp.py

## Save Results to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy checkpoints to Drive
import os

drive_path = '/content/drive/MyDrive/echelon_checkpoints'
os.makedirs(drive_path, exist_ok=True)

# Copy all .pt files
!cp *.pt {drive_path}/ 2>/dev/null || echo 'No checkpoints found yet'
print(f"Checkpoints saved to {drive_path}")